In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('creditcard.csv')

# Show basic info
print("Shape of dataset:", df.shape)
print("\nColumns:", df.columns.tolist())

# Display first 5 rows
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'creditcard.csv'

In [ ]:
#visualize the class imbalance
import matplotlib.pyplot as plt

df['Class'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Class Distribution')
plt.xticks(ticks=[0, 1], labels=['Non-Fraud (0)', 'Fraud (1)'], rotation=0)
plt.ylabel('Number of Transactions')
plt.show()


In [ ]:
df.describe()


In [4]:
from sklearn.preprocessing import StandardScaler

# Create copies to preserve original
scaled_df = df.copy()

# Initialize the scaler
scaler = StandardScaler()

# Apply scaling to 'Time' and 'Amount'
scaled_df[['Time', 'Amount']] = scaler.fit_transform(df[['Time', 'Amount']])

# Features (X) and target (y)
X = scaled_df.drop('Class', axis=1)
y = scaled_df['Class']




In [5]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

# Define resampling steps
over = SMOTE(sampling_strategy=0.1, random_state=42)  # Minority class = 10% of majority
under = RandomUnderSampler(sampling_strategy=0.5, random_state=42)  # Majority class = 2x minority

# Combine using pipeline
resample_pipeline = Pipeline(steps=[('o', over), ('u', under)])

# Apply to training set only
X_resampled, y_resampled = resample_pipeline.fit_resample(X, y)


In [ ]:
from collections import Counter

print("Original class distribution:", Counter(y))
print("Resampled class distribution:", Counter(y_resampled))


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Base model
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

# Minimal grid for quick tuning
param_grid = {
    'n_estimators': [100],
    'max_depth': [None, 10],
    'max_features': ['sqrt']
}

# Grid search with 3-fold CV
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2
)

# Train on full resampled data
grid_search_rf.fit(X_resampled, y_resampled)

# Best model
best_rf_model = grid_search_rf.best_estimator_

# Show best params
print("Best RF parameters:", grid_search_rf.best_params_)



Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best RF parameters: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 100}


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Get input dimension
input_dim = X_resampled.shape[1]

# Build the model
dnn_model = Sequential([
    Dense(128, input_dim=input_dim, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

# Compile
dnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Fit the model
history = dnn_model.fit(
    X_resampled, y_resampled,
    validation_split=0.2,
    epochs=20,
    batch_size=256,
    callbacks=[early_stop],
    verbose=2
)


c:\Users\MK\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
267/267 - 5s - 18ms/step - accuracy: 0.9545 - loss: 0.1323 - val_accuracy: 0.9144 - val_loss: 0.2147
Epoch 2/20
267/267 - 2s - 6ms/step - accuracy: 0.9806 - loss: 0.0562 - val_accuracy: 0.9307 - val_loss: 0.1516
Epoch 3/20
267/267 - 2s - 6ms/step - accuracy: 0.9852 - loss: 0.0430 - val_accuracy: 0.9652 - val_loss: 0.0903
Epoch 4/20
267/267 - 2s - 6ms/step - accuracy: 0.9879 - loss: 0.0330 - val_accuracy: 0.9780 - val_loss: 0.0640
Epoch 5/20
267/267 - 1s - 6ms/step - accuracy: 0.9906 - loss: 0.0270 - val_accuracy: 0.9873 - val_loss: 0.0461
Epoch 6/20
267/267 - 2s - 6ms/step - accuracy: 0.9921 - loss: 0.0227 - val_accuracy: 0.9947 - val_loss: 0.0243
Epoch 7/20
267/267 - 2s - 6ms/step - accuracy: 0.9937 - loss: 0.0189 - val_accuracy: 0.9970 - val_loss: 0.0229
Epoch 8/20
267/267 - 2s - 7ms/step - accuracy: 0.9949 - loss: 0.0160 - val_accuracy: 0.9970 - val_loss: 0.0164
Epoch 9/20
267/267 - 1s - 5ms/step - accuracy: 0.9952 - loss: 0.0148 - val_accuracy: 0.9992 - val_loss: 0.0144
